# Variant D — Smooth Cosine SNR Curriculum

Continuously anneals the training-SNR half-width from ±5 dB to ±10 dB along a half-cosine over 300 epochs, removing the discrete phase-boundary spikes seen in Variant 4.

This notebook is a standalone, simplified version of `varD_smooth_curriculum.py` and follows the same flow as `4 feb/ADJSCC-CSInet+.ipynb`:
1. dataset
2. AF module
3. ATN module
4. encoder
5. real → complex symbols + power normalisation
6. wireless channel
7. complex → real (C2R)
8. decoder
9. STN
10. training loop


## Imports and seed

In [ ]:
import math
import os
import random
import time
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## Config

In [ ]:
class TrainingConfig:
    train_file:        str   = "train_data.mat"
    val_file:          str   = "val_data.mat"
    test_file:         str   = "test_data.mat"
    checkpoint_dir:    str   = "checkpoints_smooth_curriculum"
    fast_dev_run:      bool  = False
    run_training:      bool  = False
    epochs:            int   = 500
    batch_size:        int   = 400          # from Var1 (stable)
    learning_rate:     float = 0.001
    min_lr:            float = 0.0001
    patience:          int   = 20
    weight_decay:      float = 1e-5
    grad_clip:         float = 1.0
    snr_center:        float = 0.0         # center of SNR range (always 0)
    snr_final_half:    float = 10.0        # full range: center ± 10
    snr_initial_half:  float = 5.0         # start range: center ± 5
    T_anneal:          int   = 300         # epochs to reach full range
    k_feedback:        int   = 64
    compression_ratio: int   = 16
    mse_weight:        float = 0.5
    nmse_weight:       float = 0.5
    warmup_epochs:     int   = 30
    save_every:        int   = 10
    run_evaluation:    bool  = False
    checkpoint_path:   str | None = None
    resume_latest:     bool  = False
cfg = TrainingConfig()
os.makedirs(cfg.checkpoint_dir, exist_ok=True)


## Dataset

Streams the QuaDRiGa CSI HDF5 files lazily and applies a global per-channel scale computed on the train split.

In [ ]:
def _read_scalar(d):
    v=d[()]; return float(v.reshape(-1)[0]) if isinstance(v,np.ndarray) and v.size==1 else v

def load_dataset_cfg(path):
    with h5py.File(path,"r") as f:
        g=f["cfg"]; return {k:_read_scalar(g[k]) for k in g if isinstance(g[k],h5py.Dataset)}

def get_train_global_scale(path, chunk=500):
    stats={"dl":{"sum_sq":0.,"count":0},"ul":{"sum_sq":0.,"count":0}}
    with h5py.File(path,"r") as f:
        for key,sn in [("csi_dl","dl"),("csi_ul","ul")]:
            ds=f[key]; N=ds.shape[3]
            for s in range(0,N,chunk):
                c=ds[:,:,:,s:min(s+chunk,N)]
                r=c["real"].astype(np.float32); im=c["imag"].astype(np.float32)
                stats[sn]["sum_sq"]+=float(np.sum(r**2)+np.sum(im**2))
                stats[sn]["count"]+=r.size+im.size
    out={k:{"std":float(np.sqrt(stats[k]["sum_sq"]/max(stats[k]["count"],1)+1e-12))} for k in ("dl","ul")}
    print("Scale:",out); return out

class CSIDatasetManager:
    def __init__(self,tp,vp,tsp,stats):
        self.stats=stats; self.files={}; self.datasets={}; self.lengths={}
        for split,path in [("train",tp),("val",vp),("test",tsp)]:
            h=h5py.File(path,"r"); self.files[split]=h
            self.datasets[split]={"dl":h["csi_dl"],"ul":h["csi_ul"]}
            self.lengths[split]=int(h["csi_dl"].shape[3]); print(f"{split}: {self.lengths[split]}")
    def close(self):
        for h in self.files.values(): h.close()
    def _norm(self,a,k): return a/(self.stats[k]["std"]+1e-8)
    def denorm(self,t,k): return t*(self.stats[k]["std"]+1e-8)
    def _proc(self,a,k,norm=True):
        r=a["real"].astype(np.float32); im=a["imag"].astype(np.float32)
        if norm: r=self._norm(r,k); im=self._norm(im,k)
        return np.transpose(np.squeeze(np.stack([r,im],2),3),(3,2,0,1))
    def get_batch(self,split,idx,sv=None,lo=None,hi=None):
        idx=np.sort(np.asarray(idx,dtype=np.int64))
        dl=torch.from_numpy(self._proc(self.datasets[split]["dl"][:,:,:,idx],"dl")).float()
        ul=torch.from_numpy(self._proc(self.datasets[split]["ul"][:,:,:,idx],"ul")).float()
        if sv is None:
            lo=lo if lo is not None else -10.; hi=hi if hi is not None else 10.
            sv=np.random.uniform(lo,hi,(len(idx),1)).astype(np.float32)
        else: sv=np.asarray(sv,dtype=np.float32).reshape(len(idx),1)
        return dl,ul,torch.from_numpy(sv).float()
    def iterate(self,split,bs,shuffle=False,rng=None,fixed_snr=None,snr_lo=None,snr_hi=None):
        N=self.lengths[split]; order=np.arange(N,dtype=np.int64)
        if shuffle: (rng if rng else np.random.default_rng()).shuffle(order)
        for s in range(0,N,bs):
            bi=order[s:s+bs]
            sv=None if fixed_snr is None else np.full((len(bi),1),fixed_snr,dtype=np.float32)
            yield self.get_batch(split,bi,sv=sv,lo=snr_lo,hi=snr_hi)

In [ ]:
# Set these paths to the QuaDRiGa CSI .mat files on your machine.
train_file = "train_data.mat"
val_file   = "val_data.mat"
test_file  = "test_data.mat"


In [ ]:
stats = get_train_global_scale(train_file)
dataset = CSIDatasetManager(train_file, val_file, test_file, stats)


## AF Module

Channel-wise SNR-aware feature recalibration: GAP over (H,W), concat with SNR (dB), 2-layer MLP → sigmoid → per-channel scale.

In [ ]:
class AFModule(nn.Module):
    def __init__(self,ch,r=2):
        super().__init__(); h=max(ch//r,1)
        self.fc1=nn.Linear(ch+1,h); self.fc2=nn.Linear(h,ch)
    def forward(self,x,snr):
        p=F.adaptive_avg_pool2d(x,1).flatten(1)
        s=torch.sigmoid(self.fc2(F.relu(self.fc1(torch.cat([p,snr],1)))))
        return x*s.view(x.size(0),x.size(1),1,1)

## ATN — Analysis Transform Network

Three-layer (or wider, in deeper variants) conv stack with asymmetric strides that compresses the 32×32 angular-delay map to the truncated representation used by the SC-CSI encoder.

In [ ]:
class ATN(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1=nn.Conv2d(2,16,3,stride=(2,1),padding=1); self.b1=nn.BatchNorm2d(16); self.p1=nn.PReLU(); self.a1=AFModule(16)
        self.c2=nn.Conv2d(16,16,3,stride=(2,1),padding=1); self.b2=nn.BatchNorm2d(16); self.p2=nn.PReLU(); self.a2=AFModule(16)
        self.c3=nn.Conv2d(16,2,3,stride=(2,1),padding=1); self.b3=nn.BatchNorm2d(2)
    def forward(self,x,snr):
        x=self.a1(self.p1(self.b1(self.c1(x))),snr)
        x=self.a2(self.p2(self.b2(self.c2(x))),snr); return self.b3(self.c3(x))

## Encoder — CSINet+ encoder with AF modules

Two 7×7 conv blocks with AF modules, then a fully-connected layer projects the flattened map to the M-dimensional real-valued codeword.

In [ ]:
class CsiNetPlusEncoderWithAF(nn.Module):
    def __init__(self,cr):
        super().__init__(); self.M=(2*32*32)//cr
        self.c1=nn.Conv2d(2,2,7,padding=3); self.b1=nn.BatchNorm2d(2); self.a1=AFModule(2)
        self.c2=nn.Conv2d(2,2,7,padding=3); self.b2=nn.BatchNorm2d(2); self.a2=AFModule(2)
        self.fc=nn.Linear(2*32*32,self.M)
    def forward(self,x,snr):
        x=self.a1(F.leaky_relu(self.b1(self.c1(x)),0.3),snr)
        x=self.a2(F.leaky_relu(self.b2(self.c2(x)),0.3),snr); return self.fc(x.flatten(1))

## Real → complex symbols + power normalisation

Splits the M real outputs into an M/2-length complex vector and rescales it to unit average power per symbol.

In [ ]:
def enc2cplx(e):
    k=e.shape[1]//2; s=torch.complex(e[:,:k],e[:,k:])
    return s/torch.sqrt(torch.mean(s.abs().square(),1,keepdim=True)+1e-8)

## Wireless channel

Differentiable OFDM AWGN channel: picks `k` uplink subcarriers, transmits the complex symbols, adds Gaussian noise scaled to the requested SNR, and applies maximum-ratio combining at the BS.

In [ ]:
class WirelessChannel(nn.Module):
    def __init__(self,Nt=32,trs=True): super().__init__(); self.Nt=Nt; self.trs=trs
    def _idx(self,ns,k,dev):
        if self.training and self.trs: return torch.randperm(ns,device=dev)[:k]
        return torch.linspace(0,ns-1,k,device=dev).round().long()
    def forward(self,s,snr,h):
        bs,k=s.shape; dev=s.device; idx=self._idx(h.shape[2],k,dev)
        hs=h[:,:,idx,:]; hu=torch.complex(hs[:,0],hs[:,1])
        nstd=torch.sqrt(1./torch.pow(10.,snr/10.)/2.).unsqueeze(-1)
        z=torch.complex(torch.randn(bs,k,self.Nt,device=dev)*nstd,
                        torch.randn(bs,k,self.Nt,device=dev)*nstd)
        y=hu*s.unsqueeze(-1)+z; w=hu/(torch.norm(hu,2,keepdim=True)+1e-8)
        return torch.sum(torch.conj(w)*y,2)

## C2R — Complex → real for the decoder

In [ ]:
class C2R(nn.Module):
    def forward(self,s): return torch.cat([s.real,s.imag],1)

## Decoder — CSINet+ RefineNet stack

FC → 32×32 feature map, an initial conv block, then a chain of RefineNet residual blocks (each conv block is followed by an AF module).

In [ ]:
class RefineBlock(nn.Module):
    def __init__(self,ch):
        super().__init__()
        self.c1=nn.Conv2d(ch,8,7,padding=3); self.b1=nn.BatchNorm2d(8); self.a1=AFModule(8)
        self.c2=nn.Conv2d(8,16,5,padding=2); self.b2=nn.BatchNorm2d(16); self.a2=AFModule(16)
        self.c3=nn.Conv2d(16,ch,3,padding=1); self.b3=nn.BatchNorm2d(ch); self.a3=AFModule(ch)
    def forward(self,x,snr):
        r=x; x=self.a1(F.leaky_relu(self.b1(self.c1(x)),0.3),snr)
        x=self.a2(F.leaky_relu(self.b2(self.c2(x)),0.3),snr); x=self.a3(self.b3(self.c3(x)),snr)
        return r+x

In [ ]:
class Decoder(nn.Module):
    def __init__(self,dim,blocks=5):
        super().__init__(); flat=2*32*32
        self.fc=nn.Linear(dim,flat)
        self.ic=nn.Conv2d(2,2,7,padding=3); self.ib=nn.BatchNorm2d(2); self.ia=AFModule(2)
        self.chain=nn.ModuleList([RefineBlock(2) for _ in range(blocks)])
    def forward(self,x,snr):
        x=self.fc(x).view(-1,2,32,32)
        x=self.ia(F.leaky_relu(self.ib(self.ic(x)),0.3),snr)
        for b in self.chain: x=b(x,snr)
        return x

## STN — Synthesis Transform Network

Mirror image of the ATN: transposed-conv stack that expands the latent back to the 32×32 angular-delay map.

In [ ]:
class STN(nn.Module):
    def __init__(self):
        super().__init__()
        self.t1=nn.ConvTranspose2d(2,16,3,stride=(2,1),padding=1,output_padding=(1,0)); self.b1=nn.BatchNorm2d(16); self.p1=nn.PReLU(); self.a1=AFModule(16)
        self.t2=nn.ConvTranspose2d(16,16,3,stride=(2,1),padding=1,output_padding=(1,0)); self.b2=nn.BatchNorm2d(16); self.p2=nn.PReLU(); self.a2=AFModule(16)
        self.t3=nn.ConvTranspose2d(16,2,3,stride=(2,1),padding=1,output_padding=(1,0)); self.b3=nn.BatchNorm2d(2)
    def forward(self,x,snr):
        x=self.a1(self.p1(self.b1(self.t1(x))),snr)
        x=self.a2(self.p2(self.b2(self.t2(x))),snr); return self.b3(self.t3(x))

## Build the modules

In [ ]:
dataset_cfg = load_dataset_cfg(train_file)
atn = ATN().to(device)
encoder = CsiNetPlusEncoderWithAF(compression_ratio=cfg.compression_ratio).to(device)
channel_sim = WirelessChannel(num_bs_antennas=int(dataset_cfg['num_bs_antennas'])).to(device)
c2r = C2R().to(device)
decoder = Decoder(input_dim=encoder.M).to(device)
stn = STN().to(device)

all_params = (list(atn.parameters()) + list(encoder.parameters())
              + list(decoder.parameters()) + list(stn.parameters()))
print('Total trainable parameters:', sum(p.numel() for p in all_params if p.requires_grad))


## Training loop

In [ ]:
def run_epoch(split,ep=0,fixed_snr=None):
    is_train=split=="train"
    for m in [atn,encoder,decoder,stn,ch_sim]: m.train(is_train)
    rng=np.random.default_rng(SEED+ep); tl=tm=ts=es=ps=0.
    # Get smooth curriculum SNR range for this epoch
    snr_lo, snr_hi = (get_snr_range(ep) if is_train
                      else (cfg.snr_center-cfg.snr_final_half, cfg.snr_center+cfg.snr_final_half))
    for bi,(Hd,Hu,snr) in enumerate(
        dataset.iterate(split,cfg.batch_size,shuffle=is_train,
                        rng=rng if is_train else None,
                        fixed_snr=fixed_snr,snr_lo=snr_lo,snr_hi=snr_hi)):
        if is_train and DBT and bi>=DBT: break
        if not is_train and DBE and bi>=DBE: break
        Hd=Hd.to(device,non_blocking=True); Hu=Hu.to(device,non_blocking=True); snr=snr.to(device,non_blocking=True)
        if is_train: optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(is_train):
            Hhat=fwd(Hd,Hu,snr); loss,ml,nl=loss_fn(Hd,Hhat,ep)
            if is_train:
                loss.backward()
                if cfg.grad_clip>0: nn.utils.clip_grad_norm_(all_params,cfg.grad_clip)
                optimizer.step()
        b=Hd.size(0); tl+=float(loss.detach())*b; tm+=float(ml)*b; ts+=b
        Hdd=dataset.denorm(Hd.detach(),"dl"); Hhd=dataset.denorm(Hhat.detach(),"dl")
        es+=float(torch.sum((Hdd-Hhd)**2)); ps+=float(torch.sum(Hdd**2))
    return {"loss":tl/max(ts,1),"mse":tm/max(ts,1),"nmse_db":nmse_db(es,ps),
            "linear_nmse":es/max(ps,1e-12),"samples":int(ts)}

def get_snr_range(epoch: int):
    hw = snr_half_width(epoch)
    return cfg.snr_center - hw, cfg.snr_center + hw

def snr_half_width(epoch: int) -> float:
    """
    Cosine annealing from initial_half to final_half over T_anneal epochs.
    Returns the half-width of the SNR range at the given epoch.
    """
    if epoch >= cfg.T_anneal:
        return cfg.snr_final_half
    progress = epoch / cfg.T_anneal
    # cosine schedule: slow start, fast middle, slow end
    frac = 0.5 * (1.0 - math.cos(math.pi * progress))
    return cfg.snr_initial_half + (cfg.snr_final_half - cfg.snr_initial_half) * frac

### Optimiser, scheduler and loss weights

These cells reproduce the variant's exact training recipe — open the source `.py` for the line-by-line argparse / CLI logic.

In [ ]:
# optimizer = optim.Adam(all_params, lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
# scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
#                                                   factor=0.5, patience=cfg.patience,
#                                                   min_lr=cfg.min_lr)
# mse_criterion = nn.MSELoss()
# (See the .py for any variant-specific overrides — e.g. cosine LR
#  schedules, AdamW, or per-parameter-group weight decay.)


### Run training

```python
for epoch in range(cfg.epochs):
    train_metrics = run_epoch('train', epoch_index=epoch)
    val_metrics   = run_epoch('val',   epoch_index=epoch)
    # scheduler.step(val_metrics['linear_nmse'])
```

After training, sweep test NMSE over a fixed SNR grid:

```python
snr_points = [-10, -5, 0, 5, 10]
nmse_db = evaluate_snr_sweep(snr_points, split='test')
```